In [15]:
# Libraries
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Layer, LeakyReLU, ReLU, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.initializers import HeNormal, GlorotNormal
from tensorflow.keras.callbacks import LearningRateScheduler
from tensorflow.keras.regularizers import l1_l2
import numpy as np

from keras.regularizers import l1, l2
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold

from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [16]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import pandas as pd
import json
# Import config variable
with open("../../config.json") as f:
    cfg = json.load(f)
    
target = cfg["rc"]        
file_name = cfg["rc_file_name"]   
random_state = cfg["split"]["random_state"]  
# Load the dataset
data = pd.read_csv(f"../../data/{file_name}.csv")

In [17]:

X = data.drop(columns=[target])
y = data[target]

X_scaler = MinMaxScaler()
y_scaler = MinMaxScaler()

In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Chargement
data = pd.read_csv('../../data/rc_training_data.csv')

# Split features / target
X = data.drop(columns=['CS (Mpa)'])
y = data['CS (Mpa)']

# Split train / test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scaling
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

In [14]:
from tensorflow.keras.initializers import HeNormal, GlorotNormal

k_folds_train = 5
kf = KFold(n_splits=k_folds_train, shuffle=True, random_state=42)

init = GlorotNormal()
l1_strength = 0.001
l2_strength = 0.01

early_stopping = EarlyStopping(patience=20, restore_best_weights=True)

mse_scores_train = []
mse_scores_val = []


r_squareds_train_all = []

r_squareds_val_all = []

models = []
historys = []

r_batch = []
batch_sizes = [4, 7, 10, 13, 16]
# batch = 14
# lr = 0.001
learning_rates = [0.0003, 0.0005, 0.0008, 0.001, 0.002]

r_squareds_train_lr = []
r_squareds_val_lr = []
mse_scores_train_lr = []
mse_scores_val_lr = []

for lr in learning_rates:
    
    for batch in batch_sizes :
        r_squareds_train_batch = []
        r_squareds_val_batch = []
        mse_scores_train_batch = []
        mse_scores_val_batch = []

        ann_rf = Sequential([
        Dense(32, activation= 'relu', kernel_initializer=init),
        Dense(16, activation= 'relu', kernel_initializer=init),
        Dropout(0.2),
        Dense(1, activation='linear', kernel_regularizer= l2(l2_strength))
        ])

        optimizer = tf.keras.optimizers.Adam(learning_rate= lr, decay = 0.01) # , decay = 0.01

        ann_rf.compile(optimizer=optimizer, 
                loss= 'mse', 
                metrics=['mae'])

        for train_index, val_index in kf.split(X_train_scaled):
            X_train_fold, X_val_fold = X_train_scaled.iloc[train_index], X_train_scaled.iloc[val_index]
            y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]

            # lr_scheduler_callback = LearningRateScheduler(lr_scheduler)

            history_train = ann_rf.fit(X_train_fold, y_train_fold, epochs=100, batch_size=batch, verbose=0)
            historys.append(history_train)

            y_pred_train = ann_rf.predict(X_train_fold)
            r_squared_train = r2_score(y_train_fold, y_pred_train)
            r_squareds_train_batch.append(r_squared_train)
            mse_train = mean_squared_error(y_train_fold, y_pred_train)
            mse_scores_train_batch.append(mse_train)

            y_pred_val = ann_rf.predict(X_val_fold)
            r_squared_val = r2_score(y_val_fold, y_pred_val)
            r_squareds_val_batch.append(r_squared_val)
            mse_val = mean_squared_error(y_val_fold, y_pred_val)
            mse_scores_val_batch.append(mse_val)

        r_squareds_train_lr.append(r_squareds_train_batch)
        r_squareds_val_lr.append(r_squareds_val_batch)
        mse_scores_train_lr.append(mse_scores_train_batch)
        mse_scores_val_lr.append(mse_scores_val_batch)
        models.append(ann_rf)
        print('Model indice :', len(models)-1)

        print('With Learning rate =', lr,"With batch =", batch, "R²_train =", np.mean(r_squareds_train_batch))
        print('With Learning rate =', lr,"With batch =", batch, "mse_train =", np.mean(mse_scores_train_batch))
        print('')
        print('With Learning rate =', lr,"With batch =", batch, "R²_val =", np.mean(r_squareds_val_batch))
        print('With Learning rate =', lr,"With batch =", batch, "mse_val =", np.mean(mse_scores_val_batch))
        print('')
        y_pred_test= ann_rf.predict(X_test_scaled)
        # y_pred_test = y_scaler.inverse_transform(y_pred_test_scaled)
        r2_rc = r2_score(y_test, y_pred_test)

        print('With Learning rate =', lr, 'With batch =', batch, "R-squared on Test Set:", r2_rc)
            #MSE
        mae_rc = mean_absolute_error(y_test, y_pred_test)

        mse_test = mean_squared_error(y_test, y_pred_test)

        print('With Learning rate =', lr,'With batch =', batch, 'Mean Absolute Error test:', mae_rc)
        print('With Learning rate =', lr,'With batch =', batch, 'Mean Squared Error test:', mse_test)
        print('With Learning rate =', lr,'With batch =', batch, 'Root Mean Squared Error test:', np.sqrt(mse_test))
        print('')
        print('With Learning rate =', lr,'With batch_size = ', batch, 'Act vs Pred :')

C:\Users\matte\AppData\Roaming\Python\Python313\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
Model indice : 0


NameError: name 'np' is not defined

In [ ]:
import joblib
import os

os.makedirs('saved_models', exist_ok=True)

# Sauvegarde du meilleur ANN (le dernier entraîné ici, adapte si tu veux le meilleur)
ann_rf.save(f'saved_models/ann_lr{lr}_batch{batch}.keras')

# Sauvegarde du scaler (indispensable pour réutiliser le modèle)
joblib.dump(scaler, 'saved_models/scaler_rc.pkl')